In [ ]:
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from co2sat.utils import data_dir
import plotly.express as px

In [ ]:
con = duckdb.connect()
p = str(data_dir("processed", "dynamic_features.parquet"))

# 100K-row random sample for distribution plots
sample = con.execute(f"""
    SELECT * FROM read_parquet('{p}') USING SAMPLE 100000 ROWS
""").df()

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Bands 1-6 — reflectance", "Bands 7-16 — brightness temp (K)"),
)
for b in range(1, 7):
    fig.add_trace(
        go.Histogram(
            x=sample[f"band_{b:02d}"], name=f"C{b:02d}", opacity=0.55, nbinsx=60
        ),
        row=1,
        col=1,
    )
for b in range(7, 17):
    fig.add_trace(
        go.Histogram(
            x=sample[f"band_{b:02d}"], name=f"C{b:02d}", opacity=0.55, nbinsx=60
        ),
        row=1,
        col=2,
    )
fig.update_layout(barmode="overlay", height=450, width=1100)
fig.show()

In [ ]:
diurnal = con.execute(f"""
    SELECT hour,
    {", ".join(f"AVG(band_{b:02d}) AS b{b:02d}" for b in range(1, 17))}
    FROM read_parquet('{p}')
    GROUP BY hour ORDER BY hour
""").df()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Reflectance bands — fleet mean by hour",
        "Thermal bands — fleet mean by hour",
    ),
)
for b in range(1, 7):
    fig.add_trace(
        go.Scatter(
            x=diurnal["hour"],
            y=diurnal[f"b{b:02d}"],
            mode="lines+markers",
            name=f"C{b:02d}",
        ),
        row=1,
        col=1,
    )
for b in range(7, 17):
    fig.add_trace(
        go.Scatter(
            x=diurnal["hour"],
            y=diurnal[f"b{b:02d}"],
            mode="lines+markers",
            name=f"C{b:02d}",
        ),
        row=1,
        col=2,
    )
fig.update_xaxes(title_text="Hour (UTC)")
fig.update_layout(height=450, width=1100)
fig.show()

In [ ]:
seasonal = con.execute(f"""
    SELECT CASE WHEN MONTH(date) IN (4, 5) THEN 'April' ELSE 'September' END AS season,
        hour,
        AVG(band_02) AS b02, AVG(band_07) AS b07,
        AVG(band_10) AS b10, AVG(band_13) AS b13
    FROM read_parquet('{p}')
    GROUP BY season, hour ORDER BY season, hour
""").df()

In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "C02 — Red visible (reflectance)",
        "C07 — Shortwave IR 3.9 μm (K)",
        "C10 — Low-level water vapor (K)",
        "C13 — Clean IR window (K)",
    ),
    vertical_spacing=0.12,
)

band_positions = {"b02": (1, 1), "b07": (1, 2), "b10": (2, 1), "b13": (2, 2)}
season_style = {
    "April": dict(color="#2a78d6"),
    "September": dict(color="#e07b39"),
}

for season in ["April", "September"]:
    sub = seasonal[seasonal["season"] == season]
    for col, (r, c) in band_positions.items():
        fig.add_trace(
            go.Scatter(
                x=sub["hour"],
                y=sub[col],
                mode="lines+markers",
                name=season,
                line=season_style[season],
                legendgroup=season,
                showlegend=(r == 1 and c == 1),  # one legend entry per season
            ),
            row=r,
            col=c,
        )

fig.update_xaxes(title_text="Hour (UTC)", row=2)
fig.update_layout(
    height=650,
    width=1000,
    title="Fleet-mean diurnal cycle: April vs September",
)
fig.show()

In [ ]:
corr = sample[[f"band_{b:02d}" for b in range(1, 17)]].corr()
fig = go.Figure(
    go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns, colorscale="RdBu", zmid=0)
)
fig.update_layout(height=600, width=700, title="Cross-band correlation (100K sample)")
fig.show()

There is a promising insight of using other satellite with less bands as the goes16 bands are severely inter-correlated

By Wien's law, emission from very hot sources peaks far closer to 3.9 μm than to the 10-12 μm window bands. Hence, we choose to look for correlation between band 07 reflectance and CO2 emissions. 

In [ ]:
epa = str(data_dir("processed", "epa_daily_with_attributes.parquet"))
signal = con.execute(f"""
    WITH daily_sat AS (
        SELECT facility_id, date, AVG(band_07) AS b07_mean
        FROM read_parquet('{p}') GROUP BY facility_id, date
    )
    SELECT s.b07_mean, e.co2_metric_tons, e.fuel_category
    FROM daily_sat s
    JOIN read_parquet('{epa}') e USING (facility_id, date)
    WHERE e.co2_metric_tons > 0
    USING SAMPLE 50000 ROWS
""").df()

In [ ]:
px.scatter(
    signal,
    x="b07_mean",
    y="co2_metric_tons",
    color="fuel_category",
    log_y=True,
    opacity=0.3,
    height=500,
).show()
print(signal[["b07_mean", "co2_metric_tons"]].corr())

Band 07 correlation with CO₂ is weak (-0.05), and that's fine; The model to be implemented earns its keep from multivariate temporal patterns.